# 🥉 Bronze Layer: Raw Data Ingestion

## What is the Bronze Layer?

In the **Medallion Architecture**, the Bronze layer is the first stage where we ingest **raw data exactly as it comes from the source** with minimal transformation.

### Key Characteristics:
* **Immutable**: Data is never modified after ingestion
* **Audit trail**: We add metadata columns to track when and how data arrived
* **Schema-on-read**: We preserve the original structure
* **Source of truth**: This is our backup if anything goes wrong downstream

### What We'll Do:
1. Create Unity Catalog schemas for Bronze, Silver, and Gold layers
2. Download Netflix titles dataset (CSV format)
3. Load raw data into a Delta Lake table
4. Add ingestion metadata for tracking

---

**Pipeline Flow:** Bronze (Raw) → Silver (Clean) → Gold (Analytics)

In [0]:
# ===================================================================
# STEP 1: Create Unity Catalog Structure
# ===================================================================
# Unity Catalog organizes data in a 3-level namespace:
# Catalog > Schema > Table
#
# We'll use the 'workspace' catalog and create 3 schemas:
# - netflix_bronze: Raw ingested data
# - netflix_silver: Cleaned and transformed data
# - netflix_gold: Business analytics aggregations
# ===================================================================

# Create Bronze schema (raw data)
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.netflix_bronze")
print("✓ Created schema: workspace.netflix_bronze")

# Create Silver schema (cleaned data)
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.netflix_silver")
print("✓ Created schema: workspace.netflix_silver")

# Create Gold schema (analytics)
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.netflix_gold")
print("✓ Created schema: workspace.netflix_gold")

print("\n✅ Unity Catalog structure ready!")

## 📥 Data Source: Netflix Titles Dataset

We'll use the **Netflix Movies and TV Shows** dataset which contains:
* Title information (name, type, director, cast)
* Content metadata (country, release year, rating)
* Temporal data (date added to Netflix)
* Categorization (genres, duration)

This dataset is publicly available and perfect for learning data engineering patterns!

In [0]:
# ===================================================================
# STEP 2: Download Netflix Titles CSV
# ===================================================================
# We're downloading from a public GitHub repository.
# In production, you'd typically read from:
# - Cloud storage (S3, ADLS, GCS)
# - External databases
# - Streaming sources (Kafka, Event Hubs)
# - APIs
# ===================================================================

import urllib.request
import os

# Public dataset URL (alternative source)
url = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2021/2021-04-20/netflix_titles.csv"

# Use /Workspace path (required for Shared Access Mode)
workspace_path = "/Workspace/Users/alexfoo29125@gmail.com/databricks_netflix"
local_path = f"{workspace_path}/netflix_titles.csv"

print(f"Downloading dataset from:\n{url}\n")

# Create directory if it doesn't exist
os.makedirs(workspace_path, exist_ok=True)

# Download the file
urllib.request.urlretrieve(url, local_path)

print(f"✅ Successfully downloaded to: {local_path}")

## 🥉 Bronze Layer Best Practices

### Why Store Raw Data?
1. **Reproducibility**: If transformations have bugs, we can reprocess from source
2. **Compliance**: Audit requirements often mandate keeping original data
3. **Flexibility**: Future use cases might need fields we initially discarded
4. **Debugging**: Compare raw vs transformed data to troubleshoot issues

### What We Add:
* `ingestion_timestamp`: When the data was loaded
* `source_file`: Which file the record came from

### Delta Lake Benefits:
* **ACID transactions**: Reliable writes, no partial data
* **Time travel**: Query historical versions
* **Schema evolution**: Add columns without breaking existing queries
* **Performance**: Optimized storage with Parquet + statistics

In [0]:
# ===================================================================
# STEP 3: Load CSV into Bronze Delta Table
# ===================================================================
# This is the core ingestion step:
# 1. Read CSV with inferred schema
# 2. Add audit/metadata columns
# 3. Write to Delta Lake format
# 4. Register as a Unity Catalog table
# ===================================================================

from pyspark.sql.functions import current_timestamp, lit

print("Reading CSV file...")

# Read CSV with automatic schema inference
df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("escape", "\"") \
    .load(local_path)

print(f"✓ Loaded {df_raw.count():,} records from CSV")
print(f"✓ Schema has {len(df_raw.columns)} columns\n")

# Add metadata columns for audit trail
# Note: Using lit() for source_file since input_file_name() is not supported in Unity Catalog Shared Access Mode
df_bronze = df_raw \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file", lit(local_path))

print("Writing to Delta Lake...")

# Write to Bronze Delta table
# mode("overwrite") - replaces existing data (use "append" for incremental loads)
df_bronze.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.netflix_bronze.raw_titles")

print(f"\n✅ Successfully loaded {df_bronze.count():,} records into Bronze layer")
print("\n📊 Table: workspace.netflix_bronze.raw_titles")

In [0]:
# ===================================================================
# STEP 4: Inspect the Bronze Table
# ===================================================================
# Let's see what the raw data looks like
# ===================================================================

print("📋 Sample records from Bronze layer:\n")

# Read from the Delta table we just created
df_bronze_table = spark.table("workspace.netflix_bronze.raw_titles")

# Display first 10 rows
display(df_bronze_table.limit(10))

In [0]:
# ===================================================================
# STEP 5: Table Metadata and Statistics
# ===================================================================
# Delta Lake tracks important metadata about tables
# ===================================================================

print("📊 Bronze Table Statistics:\n")

# Get basic stats
row_count = df_bronze_table.count()
col_count = len(df_bronze_table.columns)

print(f"Total Rows: {row_count:,}")
print(f"Total Columns: {col_count}")
print(f"\nColumn Names:")
for col in df_bronze_table.columns:
    print(f"  • {col}")

print("\n" + "="*70)
print("Delta Table Details:")
print("="*70 + "\n")

# Show Delta Lake metadata
spark.sql("DESCRIBE DETAIL workspace.netflix_bronze.raw_titles").show(truncate=False)

## ✅ Bronze Layer Complete!

### What We Accomplished:
1. ✓ Created Unity Catalog schema structure
2. ✓ Downloaded Netflix dataset from public source
3. ✓ Loaded raw CSV data into Delta Lake
4. ✓ Added audit columns (ingestion_timestamp, source_file)
5. ✓ Registered table in Unity Catalog: `workspace.netflix_bronze.raw_titles`

### Key Bronze Layer Principles:
* **Immutable**: Never modify Bronze data after ingestion
* **Complete**: All source columns preserved
* **Auditable**: Metadata tracks data lineage
* **Versioned**: Delta Lake enables time travel

---

## 🎯 Next Steps:

Now that we have raw data in Bronze, we'll move to the **Silver Layer** where we'll:
* Handle null values and missing data
* Standardize column names
* Convert date fields to proper types
* Remove duplicate records
* Derive new analytical columns (content_age, decade)
* Add data quality validations

**Continue to:** `02_Silver_Netflix_Transformation` notebook